<table>
    <tr>
        <td><img src="https://s3.amazonaws.com/media-p.slid.es/uploads/1485763/images/9060062/Header.png" width="300"/></td>
        <td>&nbsp;</td>
        <td>
            <h1 style="font-size:200%;color:#6A0DAD;text-align:center"> <FONT COLOR="#6A0DAD"> Introducción a Redes Neuronales</FONT> </h1></td>         
        <td>
            <tp><p style="font-size:99%;text-align:center">Sesión 6</p></tp>
            <tp><p style="font-size:115%;text-align:center">Mayo 2026</p></tp>
            <tp><p style="font-size:115%;text-align:center">Prof. Daniel Rambaut </p></tp>
        </td>
    </tr>
</table>

En este notebook daremos los primeros pasos en el fascinante mundo de las **Redes Neuronales Artificiales (ANN)**, la base del *Deep Learning*. Veremos cómo, inspirándose en el funcionamiento del cerebro humano, podemos construir modelos capaces de aprender patrones extremadamente complejos a partir de los datos.
<br>
Aprenderemos sobre:
- La **neurona artificial** y el **perceptrón** (la unidad mínima de cómputo).
- Las **funciones de activación** (Sigmoide, ReLU, Tanh, Softmax) y por qué son indispensables.
- La idea intuitiva detrás de **Backpropagation** y el **Descenso del Gradiente**.
- Cómo construir nuestras primeras redes neuronales usando **Keras + TensorFlow**.

---
# <FONT SIZE=5 COLOR="green"> 0. Repaso de la sesión anterior </FONT>

En la sesión pasada profundizamos en los **Modelos de Regresión**, donde el objetivo es predecir una variable numérica continua. Los conceptos clave fueron:

<FONT COLOR="green" SIZE=3>1. Regresión Lineal Simple y Múltiple:</FONT>  
Aprendimos a estimar una variable objetivo $y$ a partir de una o varias variables predictoras $X$ usando la ecuación $\hat{y} = \beta_0 + \beta_1 x_1 + \dots + \beta_n x_n$. El modelo encuentra los coeficientes $\beta$ óptimos mediante el método de **Mínimos Cuadrados**.

<FONT COLOR="green" SIZE=3>2. Métricas de evaluación:</FONT>
<br>
Vimos cómo medir la calidad de las predicciones usando el **Error Cuadrático Medio (MSE)**, su raíz **RMSE** y el coeficiente de determinación $R^2$, que nos indica qué porcentaje de la varianza explica el modelo.

<FONT COLOR="green" SIZE=3>3. Limitaciones de los modelos lineales:</FONT>
<br>
Notamos que cuando la relación entre las variables no es lineal, los modelos lineales fallan. Aquí es donde entran modelos más flexibles como los **árboles**, **SVR** y, como veremos hoy, las **Redes Neuronales**.

---

# <FONT SIZE=5 COLOR="purple"> 1. Introducción a las Redes Neuronales </FONT>

Las **Redes Neuronales Artificiales** son modelos computacionales **inspirados** (no copiados) en el funcionamiento del cerebro biológico. La idea fundamental es que millones de unidades simples, llamadas *neuronas*, se conectan entre sí para resolver problemas que serían demasiado complejos para un solo modelo lineal.

<br>

<center><FONT SIZE=4 COLOR="BLUE">Analogía: Neurona Biológica vs. Neurona Artificial </FONT></center>

<br>
<center><img src="https://upload.wikimedia.org/wikipedia/commons/1/10/Blausen_0657_MultipolarNeuron.png" alt="centered image" width="450" height="280"></center>
<center><figcaption> <FONT SIZE=1 COLOR="black"> Fuente: Wikimedia Commons - Blausen Medical </FONT></figcaption></center>
<br>

| Neurona Biológica | Neurona Artificial |
|---|---|
| **Dendritas** (reciben señales) | **Entradas** $x_1, x_2, \dots, x_n$ |
| **Sinapsis** (conexiones con peso) | **Pesos** $w_1, w_2, \dots, w_n$ |
| **Núcleo** (procesa la señal) | **Suma ponderada + Función de activación** |
| **Axón** (transmite la salida) | **Salida** $\hat{y}$ |

<br>

<FONT COLOR="purple" SIZE=3>**¿Por qué son tan importantes hoy?**</FONT>

- Pueden modelar relaciones **altamente no lineales**.
- Son la base del *Deep Learning*: visión por computador, NLP, reconocimiento de voz, generación de imágenes, etc.
- Se benefician enormemente del aumento de datos y poder computacional (GPUs/TPUs).

A continuación, definiremos las librerías necesarias para trabajar con redes neuronales.

In [ ]:
# Manejo de datos
import pandas as pd
import numpy  as np

# Librerías para graficar
import matplotlib.pyplot as plt
import seaborn           as sns

# Sklearn: utilidades de Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.datasets        import make_moons, make_classification, load_breast_cancer
from sklearn.metrics         import accuracy_score, classification_report, confusion_matrix, mean_squared_error, r2_score

# TensorFlow + Keras: nuestro framework para Redes Neuronales
import tensorflow as tf
from tensorflow            import keras
from tensorflow.keras      import layers, models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

# Warnings
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)

# <FONT SIZE=5 COLOR="purple"> 2. La Neurona Artificial y el Perceptrón </FONT>

## <FONT SIZE=4 COLOR="green"> 2.1 La Neurona Artificial </FONT>

Una **neurona artificial** es la unidad más básica de cómputo en una red neuronal. Recibe varias entradas, las combina mediante una **suma ponderada**, le añade un **sesgo (bias)**, y aplica una **función de activación** para producir una salida.

<br>
<center><img src="https://upload.wikimedia.org/wikipedia/commons/6/60/ArtificialNeuronModel_english.png" alt="centered image" width="600" height="280"></center>
<center><figcaption><FONT SIZE=1 COLOR="black">Fuente: Wikimedia Commons</FONT></figcaption></center>
<br>

Matemáticamente, la salida $\hat{y}$ de una neurona se calcula así:

$$ z = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b = \sum_{i=1}^n w_i x_i + b $$

$$ \hat{y} = f(z) $$

donde:

- $x_i$ son las **entradas** (features).
- $w_i$ son los **pesos** que la red debe aprender.
- $b$ es el **sesgo** (bias), un parámetro adicional que aprende la red.
- $f$ es la **función de activación** (la veremos en la próxima sección).

## <FONT SIZE=4 COLOR="green"> 2.2 El Perceptrón </FONT>

El **Perceptrón** (Frank Rosenblatt, 1958) es la versión más simple de una neurona artificial. Usa una **función escalón (step function)** como activación:

$$ \hat{y} = \begin{cases} 1 & \text{si } z \geq 0 \\ 0 & \text{si } z < 0 \end{cases} $$

Es decir, el perceptrón es un **clasificador binario lineal**: dibuja una línea recta (o un hiperplano en dimensiones mayores) y clasifica según en qué lado caiga el dato.

<FONT COLOR="red" SIZE=3>⚠️ **Limitación del Perceptrón:**</FONT> solo puede resolver problemas **linealmente separables**. El famoso problema **XOR** (1969, Minsky & Papert) demostró que un perceptrón simple no puede aprenderlo, lo que llevó al primer "invierno" de las redes neuronales. La solución llegó al **apilar** múltiples neuronas en capas: el **Perceptrón Multicapa (MLP)**.

## <FONT SIZE=4 COLOR="green"> 2.3 Implementemos una neurona desde cero </FONT>

Vamos a programar una neurona simple en NumPy para entender exactamente cómo funciona internamente.

In [ ]:
# Implementación de una neurona artificial desde cero

def neurona(entradas, pesos, sesgo, activacion):
    """
    Calcula la salida de una neurona artificial.
    
    Parámetros:
    - entradas: vector de features (x1, x2, ..., xn)
    - pesos: vector de pesos (w1, w2, ..., wn)
    - sesgo: bias (b)
    - activacion: función de activación a aplicar
    
    Retorna: la salida de la neurona
    """
    # Paso 1: suma ponderada
    z = np.dot(entradas, pesos) + sesgo
    # Paso 2: función de activación
    salida = activacion(z)
    return salida

# Función escalón (Perceptrón clásico)
def escalon(z):
    return 1 if z >= 0 else 0

# Probemos con un ejemplo
entradas = np.array([1.5, 2.0, -0.5])
pesos    = np.array([0.4, 0.3, 0.8])
sesgo    = 0.1

resultado = neurona(entradas, pesos, sesgo, escalon)
print("Entradas:", entradas)
print("Pesos:   ", pesos)
print("Sesgo:   ", sesgo)
print("Suma ponderada (z):", np.dot(entradas, pesos) + sesgo)
print("Salida de la neurona:", resultado)

## <FONT SIZE=4 COLOR="blue"> 2.4 Ejercicios — Neurona y Perceptrón </FONT>

### <FONT SIZE=3 COLOR="blue"> Ejercicio 1 </FONT>
Explica con tus propias palabras la diferencia entre los **pesos** ($w_i$) y el **sesgo** ($b$) en una neurona. ¿Qué papel cumple cada uno geométricamente al trazar la frontera de decisión?


*Escribe tu respuesta aquí:*


### <FONT SIZE=3 COLOR="blue"> Ejercicio 2 </FONT>
Investiga el problema **XOR** (función "O exclusivo"). ¿Por qué un perceptrón simple no puede resolverlo? Acompaña tu respuesta con un dibujo o esquema mental de cómo se ven los puntos en el plano.


*Escribe tu respuesta aquí:*


### <FONT SIZE=3 COLOR="blue"> Ejercicio 3 </FONT>
Modifica la función `neurona` que vimos arriba para que reciba **dos entradas** y simule una compuerta lógica **AND** (devuelve 1 solo si ambas entradas son 1, sino 0). 

**Pista:** los pesos $w_1 = w_2 = 1$ y un sesgo $b = -1.5$ funcionan. Verifica que la tabla de verdad coincide:

| $x_1$ | $x_2$ | AND |
|-------|-------|-----|
|   0   |   0   |  0  |
|   0   |   1   |  0  |
|   1   |   0   |  0  |
|   1   |   1   |  1  |

In [ ]:
# TU CÓDIGO AQUÍ:

# 1. Define los pesos y el sesgo para la compuerta AND
pesos_and = ...
sesgo_and = ...

# 2. Recorre las 4 combinaciones posibles de (x1, x2) y prueba la neurona
combinaciones = [(0,0), (0,1), (1,0), (1,1)]

print("x1 | x2 | AND")
print("-" * 18)
for x1, x2 in combinaciones:
    entrada = np.array([x1, x2])
    # Llama a la función neurona() usando la activación escalon
    salida = ...
    print(f" {x1} |  {x2} |  {salida}")

# <FONT SIZE=5 COLOR="purple"> 3. Funciones de Activación </FONT>

Las **funciones de activación** son piezas clave: introducen **no linealidad** en la red. Sin ellas, una red neuronal — sin importar cuántas capas tenga — sería equivalente a una simple regresión lineal.

<center><FONT SIZE=4 COLOR="BLUE">¿Por qué necesitamos no linealidad? </FONT></center>

Si componemos varias funciones lineales, el resultado sigue siendo lineal:
$$ f(x) = W_2(W_1 x + b_1) + b_2 = (W_2 W_1) x + (W_2 b_1 + b_2) $$

Por tanto, sin funciones de activación no lineales **no podríamos aprender patrones complejos**.

<br>

A continuación, las funciones de activación más utilizadas:

## <FONT SIZE=4 COLOR="green"> 3.1 Funciones de activación más comunes </FONT>

| Función | Fórmula | Rango | Uso típico |
|---------|---------|-------|------------|
| **Sigmoide** | $\sigma(z) = \dfrac{1}{1+e^{-z}}$ | $(0, 1)$ | Salida binaria (probabilidad) |
| **Tanh** | $\tanh(z) = \dfrac{e^z - e^{-z}}{e^z + e^{-z}}$ | $(-1, 1)$ | Capas ocultas (centrada en 0) |
| **ReLU** | $\text{ReLU}(z) = \max(0, z)$ | $[0, \infty)$ | **La más usada** en capas ocultas |
| **Leaky ReLU** | $\max(\alpha z, z)$ con $\alpha$ pequeño | $(-\infty, \infty)$ | Evita "neuronas muertas" |
| **Softmax** | $\dfrac{e^{z_i}}{\sum_j e^{z_j}}$ | $(0, 1)$, suman 1 | Capa de salida en clasificación multiclase |

<br>

<FONT COLOR="green" SIZE=3>**Reglas prácticas:**</FONT>
- **Capas ocultas:** usa **ReLU** por defecto (rápida, evita el problema del *vanishing gradient*).
- **Capa de salida — clasificación binaria:** **Sigmoide**.
- **Capa de salida — clasificación multiclase:** **Softmax**.
- **Capa de salida — regresión:** **lineal** (sin activación).

Visualicemos cómo se ven estas funciones:

In [ ]:
# Definimos las funciones de activación
def sigmoide(z):
    return 1 / (1 + np.exp(-z))

def tanh(z):
    return np.tanh(z)

def relu(z):
    return np.maximum(0, z)

def leaky_relu(z, alpha=0.1):
    return np.where(z > 0, z, alpha * z)

# Eje x
z = np.linspace(-5, 5, 200)

# Graficamos
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(z, sigmoide(z), color='purple', linewidth=2)
axes[0, 0].set_title('Sigmoide', fontsize=14, fontweight='bold')
axes[0, 0].grid(alpha=0.3); axes[0, 0].axhline(0, color='gray', lw=0.5); axes[0, 0].axvline(0, color='gray', lw=0.5)

axes[0, 1].plot(z, tanh(z), color='green', linewidth=2)
axes[0, 1].set_title('Tanh', fontsize=14, fontweight='bold')
axes[0, 1].grid(alpha=0.3); axes[0, 1].axhline(0, color='gray', lw=0.5); axes[0, 1].axvline(0, color='gray', lw=0.5)

axes[1, 0].plot(z, relu(z), color='blue', linewidth=2)
axes[1, 0].set_title('ReLU', fontsize=14, fontweight='bold')
axes[1, 0].grid(alpha=0.3); axes[1, 0].axhline(0, color='gray', lw=0.5); axes[1, 0].axvline(0, color='gray', lw=0.5)

axes[1, 1].plot(z, leaky_relu(z), color='red', linewidth=2)
axes[1, 1].set_title('Leaky ReLU', fontsize=14, fontweight='bold')
axes[1, 1].grid(alpha=0.3); axes[1, 1].axhline(0, color='gray', lw=0.5); axes[1, 1].axvline(0, color='gray', lw=0.5)

plt.suptitle('Funciones de Activación', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## <FONT SIZE=4 COLOR="blue"> 3.2 Ejercicios — Funciones de Activación </FONT>

### <FONT SIZE=3 COLOR="blue"> Ejercicio 1 </FONT>
Si yo te dijera que una red neuronal usa la función **identidad** ($f(z)=z$) en *todas* sus capas, ¿qué tipo de modelo termina siendo equivalente esta red? Justifica matemáticamente.


*Escribe tu respuesta aquí:*


### <FONT SIZE=3 COLOR="blue"> Ejercicio 2 </FONT>
La función **Sigmoide** fue muy popular en los años 90, pero hoy se prefiere **ReLU** en capas ocultas. Investiga y explica el problema del **"vanishing gradient"** (desvanecimiento del gradiente) y por qué ReLU lo mitiga.


*Escribe tu respuesta aquí:*


### <FONT SIZE=3 COLOR="blue"> Ejercicio 3 </FONT>
Para cada uno de los siguientes problemas, indica qué **función de activación** usarías en la **capa de salida** y por qué:

a) Predecir si un correo es **spam** o no.  
b) Clasificar imágenes en **10 categorías** (dígitos del 0 al 9).  
c) Predecir el **precio** de una vivienda.  
d) Clasificar si un paciente tiene una de **3 enfermedades posibles**.


*Escribe tu respuesta aquí:*


### <FONT SIZE=3 COLOR="blue"> Ejercicio 4 </FONT>
Implementa la función **Softmax** en NumPy. Recuerda la fórmula:

$$ \text{softmax}(z_i) = \dfrac{e^{z_i}}{\sum_j e^{z_j}} $$

Verifica que las salidas suman 1 (es decir, son probabilidades válidas).

In [ ]:
# TU CÓDIGO AQUÍ:

def softmax(z):
    # 1. Calcula los exponenciales de cada elemento
    exp_z = ...
    # 2. Divide por la suma de los exponenciales
    return ...

# Prueba con un vector de logits (salidas crudas de la red)
logits = np.array([2.0, 1.0, 0.1])
probs = softmax(logits)

print("Logits:        ", logits)
print("Probabilidades:", probs)
print("Suma:          ", probs.sum(), "  <-- debe dar 1.0")

# <FONT SIZE=5 COLOR="purple"> 4. Arquitectura de una Red Neuronal y Backpropagation </FONT>

## <FONT SIZE=4 COLOR="green"> 4.1 Arquitectura: capas, neuronas y conexiones </FONT>

Una **Red Neuronal Multicapa (MLP)** está compuesta por:

- **Capa de entrada (input layer):** una neurona por cada feature del dataset.
- **Capas ocultas (hidden layers):** una o varias capas intermedias donde ocurre la "magia". Cada neurona aplica una transformación no lineal.
- **Capa de salida (output layer):** produce la predicción final. El número de neuronas depende del problema (1 para regresión / clasificación binaria, $k$ para clasificación de $k$ clases).

<br>
<center><img src="https://upload.wikimedia.org/wikipedia/commons/c/c2/MultiLayerNeuralNetworkBigger_english.png" alt="centered image" width="600" height="350"></center>
<center><figcaption><FONT SIZE=1 COLOR="black">Fuente: Wikimedia Commons</FONT></figcaption></center>
<br>

Cuando la red tiene **muchas capas ocultas**, hablamos de *Deep Learning* (aprendizaje profundo).

## <FONT SIZE=4 COLOR="green"> 4.2 Forward Propagation (paso hacia adelante) </FONT>

Es el proceso de **calcular la predicción** de la red:

1. Los datos entran por la capa de entrada.
2. Cada capa calcula $z = Wx + b$ y aplica su función de activación.
3. La salida de una capa es la entrada de la siguiente.
4. La última capa produce $\hat{y}$.

$$ \hat{y} = f^{(L)}\Big( W^{(L)} \cdot f^{(L-1)}\big( W^{(L-1)} \cdot \dots \cdot f^{(1)}(W^{(1)} x + b^{(1)}) \dots \big) + b^{(L)} \Big) $$

## <FONT SIZE=4 COLOR="green"> 4.3 Función de pérdida (Loss Function) </FONT>

Una vez tenemos la predicción $\hat{y}$, debemos comparar contra el valor real $y$. La **función de pérdida** mide qué tan mal lo está haciendo la red:

| Problema | Función de pérdida | Fórmula |
|----------|-------------------|---------|
| Regresión | **MSE** | $\frac{1}{n}\sum (y_i - \hat{y}_i)^2$ |
| Clasificación binaria | **Binary Cross-Entropy** | $-\frac{1}{n}\sum \big[ y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i) \big]$ |
| Clasificación multiclase | **Categorical Cross-Entropy** | $-\sum_i \sum_c y_{i,c} \log(\hat{y}_{i,c})$ |

El objetivo del entrenamiento es **minimizar la pérdida**.

## <FONT SIZE=4 COLOR="green"> 4.4 Backpropagation: la idea clave </FONT>

**Backpropagation** (propagación hacia atrás) es el algoritmo que permite a la red **aprender** ajustando sus pesos. Se basa en dos ideas:

1. **Descenso del Gradiente:** mover los pesos en la dirección opuesta al gradiente de la función de pérdida.
$$ w \leftarrow w - \eta \cdot \frac{\partial L}{\partial w} $$

   donde $\eta$ (eta) es la **tasa de aprendizaje** (*learning rate*).

2. **Regla de la cadena:** propagar el error desde la salida hasta cada peso usando la regla de la cadena del cálculo.

<FONT COLOR="purple" SIZE=3>**Resumen del entrenamiento (un paso o "epoch"):**</FONT>

```
1. Forward pass:    calcular ŷ con los pesos actuales
2. Calcular pérdida: L(y, ŷ)
3. Backward pass:    calcular ∂L/∂w para cada peso (usando la regla de la cadena)
4. Update:           w ← w - η · ∂L/∂w
5. Repetir miles de veces hasta que L sea pequeña
```

<br>

<FONT COLOR="red" SIZE=3>⚠️ **Hiperparámetros importantes que conocerás:**</FONT>

- **Learning rate ($\eta$):** controla qué tan grandes son los pasos. Si es muy alto, oscila; si es muy bajo, tarda mucho.
- **Epochs:** cuántas veces vamos a recorrer el dataset completo.
- **Batch size:** cuántas muestras se usan en cada paso de actualización (mini-batch).
- **Optimizador:** variantes inteligentes del descenso del gradiente (SGD, Adam, RMSprop, ...). **Adam** es el más usado por defecto.

## <FONT SIZE=4 COLOR="green"> 4.5 Visualicemos el Descenso del Gradiente </FONT>

Para entender intuitivamente qué hace el descenso del gradiente, simulémoslo sobre una función simple $L(w) = w^2$. El óptimo está en $w=0$ (donde la pérdida vale 0).

In [ ]:
# Función de pérdida (parábola simple): L(w) = w^2
def L(w):
    return w**2

# Gradiente (derivada): dL/dw = 2w
def gradiente(w):
    return 2*w

# Hiperparámetros
w        = 8.0      # Peso inicial (lejos del óptimo)
lr       = 0.1      # Learning rate
n_steps  = 30       # Número de pasos

historia_w = [w]
for paso in range(n_steps):
    grad = gradiente(w)
    w = w - lr * grad   # <-- regla del descenso del gradiente
    historia_w.append(w)

# Graficamos
ws = np.linspace(-9, 9, 200)
plt.figure(figsize=(10, 6))
plt.plot(ws, L(ws), 'b-', linewidth=2, label='L(w) = w²')
plt.scatter(historia_w, [L(w) for w in historia_w], color='red', s=50, zorder=5, label='Pasos del Descenso')
for i in range(len(historia_w)-1):
    plt.annotate('', xy=(historia_w[i+1], L(historia_w[i+1])), xytext=(historia_w[i], L(historia_w[i])),
                 arrowprops=dict(arrowstyle='->', color='red', alpha=0.5))
plt.title(f'Descenso del Gradiente — learning rate = {lr}', fontsize=14, fontweight='bold')
plt.xlabel('w (peso)'); plt.ylabel('L(w) (pérdida)')
plt.grid(alpha=0.3); plt.legend()
plt.show()

print(f"Peso inicial: 8.0  -->  Peso final tras {n_steps} pasos: {w:.6f}")
print("(El óptimo es w = 0)")

## <FONT SIZE=4 COLOR="blue"> 4.6 Ejercicios — Backpropagation </FONT>

### <FONT SIZE=3 COLOR="blue"> Ejercicio 1 </FONT>
Explica con tus propias palabras la diferencia entre **forward propagation** y **backpropagation**. ¿Cuál ocurre primero y por qué?


*Escribe tu respuesta aquí:*


### <FONT SIZE=3 COLOR="blue"> Ejercicio 2 </FONT>
Imagina que estás entrenando una red y observas que la pérdida **oscila violentamente** en cada epoch sin converger. Después la entrenas con un *learning rate* mucho menor y observas que tarda **demasiadas epochs** en bajar. ¿Cómo elegirías un *learning rate* adecuado? ¿Qué técnicas conoces para ajustarlo?


*Escribe tu respuesta aquí:*


### <FONT SIZE=3 COLOR="blue"> Ejercicio 3 </FONT>
Modifica el código de descenso del gradiente que ejecutamos arriba y prueba con tres valores distintos de `lr`: **0.01**, **0.5** y **1.05**. Para cada uno, grafica la trayectoria del peso y describe qué ocurre.

**Pista:** con `lr = 1.05` el método **diverge** (los pasos se hacen cada vez más grandes).

In [ ]:
# TU CÓDIGO AQUÍ:
# Convierte el código del descenso del gradiente en una función y prueba con 3 learning rates.

def descenso_gradiente(w_inicial, lr, n_steps):
    """Devuelve la lista de pesos visitados en cada paso."""
    w = w_inicial
    historia = [w]
    for paso in range(n_steps):
        grad = ...           # Completa: gradiente de w^2
        w = ...              # Completa: actualización
        historia.append(w)
    return historia

# Prueba con tres learning rates
lrs = [0.01, 0.5, 1.05]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ws = np.linspace(-10, 10, 200)
for ax, lr in zip(axes, lrs):
    historia = descenso_gradiente(w_inicial=8.0, lr=lr, n_steps=20)
    ax.plot(ws, ws**2, 'b-', linewidth=2)
    ax.scatter(historia, [w**2 for w in historia], color='red', s=40)
    ax.set_title(f'lr = {lr}')
    ax.set_xlabel('w'); ax.set_ylabel('L(w)')
    ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

# <FONT SIZE=5 COLOR="purple"> 5. Redes Neuronales en Keras + TensorFlow </FONT>

## <FONT SIZE=4 COLOR="green"> 5.1 ¿Qué son TensorFlow y Keras? </FONT>

- **TensorFlow** es una librería *open source* desarrollada por Google para cálculo numérico y machine learning, especialmente redes neuronales. Maneja **tensores** (arreglos N-dimensionales) y aprovecha automáticamente GPU/TPU.

- **Keras** es una **API de alto nivel** que vive dentro de TensorFlow. Permite construir redes con muy pocas líneas de código.

## <FONT SIZE=4 COLOR="green"> 5.2 Pasos para construir una red en Keras </FONT>

El flujo de trabajo siempre es el mismo:

1. **Definir** la arquitectura (`Sequential` o API funcional).
2. **Compilar** la red (elegir optimizador, función de pérdida y métricas).
3. **Entrenar** con `.fit()` (especificar epochs y batch size).
4. **Evaluar** con `.evaluate()` y **predecir** con `.predict()`.

## <FONT SIZE=4 COLOR="green"> 5.3 Ejemplo práctico: Clasificación binaria con `make_moons` </FONT>

Vamos a generar un dataset clásico no lineal (forma de medias lunas) y entrenar una red neuronal para clasificarlo. Esto es un problema que **un perceptrón simple no podría resolver** (no es linealmente separable).

In [ ]:
# 1. Generamos los datos
X, y = make_moons(n_samples=1000, noise=0.20, random_state=42)

# Visualicemos
plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu', edgecolors='k', s=30)
plt.title('Dataset make_moons — clases NO linealmente separables', fontsize=12)
plt.xlabel('x1'); plt.ylabel('x2')
plt.grid(alpha=0.3); plt.show()

# 2. Train / test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Escalado (¡muy importante en redes neuronales!)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print("X_train shape:", X_train.shape)
print("X_test  shape:", X_test.shape)

In [ ]:
# Definimos la arquitectura de la red
modelo = Sequential([
    Dense(16, activation='relu', input_shape=(2,)),  # Capa oculta 1: 16 neuronas
    Dense(8,  activation='relu'),                    # Capa oculta 2: 8 neuronas
    Dense(1,  activation='sigmoid')                  # Capa de salida: 1 neurona (binario)
])

# Resumen de la arquitectura
modelo.summary()

In [ ]:
# Compilamos: optimizador + pérdida + métrica
modelo.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Entrenamos
historia = modelo.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    verbose=0
)

# Evaluamos en test
loss_test, acc_test = modelo.evaluate(X_test, y_test, verbose=0)
print(f"\nAccuracy en test: {acc_test:.4f}")
print(f"Pérdida en test:  {loss_test:.4f}")

In [ ]:
# Graficamos el historial de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(historia.history['loss'],     label='train', linewidth=2)
axes[0].plot(historia.history['val_loss'], label='val',   linewidth=2)
axes[0].set_title('Pérdida (loss)'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(historia.history['accuracy'],     label='train', linewidth=2)
axes[1].plot(historia.history['val_accuracy'], label='val',   linewidth=2)
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# Visualizamos la frontera de decisión aprendida por la red

def plot_frontera(modelo, X, y, scaler, titulo='Frontera de decisión'):
    h = 0.02
    x1_min, x1_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x2_min, x2_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx1, xx2 = np.meshgrid(np.arange(x1_min, x1_max, h),
                           np.arange(x2_min, x2_max, h))
    grid = np.c_[xx1.ravel(), xx2.ravel()]
    grid_scaled = scaler.transform(grid)
    Z = modelo.predict(grid_scaled, verbose=0).reshape(xx1.shape)
    
    plt.figure(figsize=(8, 6))
    plt.contourf(xx1, xx2, Z, levels=20, cmap='RdBu', alpha=0.7)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu', edgecolors='k', s=30)
    plt.title(titulo, fontsize=13, fontweight='bold')
    plt.xlabel('x1'); plt.ylabel('x2')
    plt.colorbar(label='P(clase=1)')
    plt.show()

# Datos originales (sin escalar) solo para graficar
X_orig, y_orig = make_moons(n_samples=1000, noise=0.20, random_state=42)
plot_frontera(modelo, X_orig, y_orig, scaler, 'Frontera aprendida por la Red Neuronal')

## <FONT SIZE=4 COLOR="green"> 5.4 Caso real: Diagnóstico de cáncer de mama </FONT>

Apliquemos lo que aprendimos a un dataset real de medicina: el **Wisconsin Breast Cancer Dataset**, que contiene mediciones de tumores y la variable objetivo es si son **malignos (0)** o **benignos (1)**.

In [ ]:
# Cargamos el dataset
data = load_breast_cancer()
X = data.data
y = data.target

print("Forma de X:", X.shape)
print("Clases:", data.target_names)
print("Distribución:", np.bincount(y))

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Escalado
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [ ]:
# Construimos una red más profunda (2 capas ocultas) con Dropout para regularizar
modelo_cancer = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

modelo_cancer.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

modelo_cancer.summary()

In [ ]:
# Entrenamos
historia_cancer = modelo_cancer.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=60,
    batch_size=16,
    verbose=0
)

# Evaluamos
loss, acc = modelo_cancer.evaluate(X_test, y_test, verbose=0)
print(f"Accuracy en test: {acc:.4f}")

# Predicciones y reporte de clasificación
y_proba = modelo_cancer.predict(X_test, verbose=0).flatten()
y_pred  = (y_proba > 0.5).astype(int)

print("\n" + classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=data.target_names, yticklabels=data.target_names)
plt.title('Matriz de Confusión — Red Neuronal'); plt.xlabel('Predicción'); plt.ylabel('Real')
plt.show()

## <FONT SIZE=4 COLOR="blue"> 5.5 Ejercicios — Keras + TensorFlow </FONT>

### <FONT SIZE=3 COLOR="blue"> Ejercicio 1 </FONT>
En el modelo del cáncer usamos la capa `Dropout`. Investiga y explica con tus propias palabras qué hace esta capa y para qué sirve. Conecta tu respuesta con el concepto de **sobreajuste**.


*Escribe tu respuesta aquí:*


### <FONT SIZE=3 COLOR="blue"> Ejercicio 2 </FONT>
Toma el modelo `modelo_cancer` y modifícalo para que tenga **3 capas ocultas** con **64, 32 y 16 neuronas** respectivamente (todas con ReLU y un Dropout de 0.3 después de cada una). Compila, entrena 50 epochs y compara la accuracy con el modelo original. ¿Mejoró?

In [ ]:
# TU CÓDIGO AQUÍ:

# 1. Define la nueva arquitectura
modelo_v2 = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    # ... completa las capas faltantes ...
    Dense(1, activation='sigmoid')
])

# 2. Compila
modelo_v2.compile(...)

# 3. Entrena durante 50 epochs
historia_v2 = ...

# 4. Evalúa en test e imprime accuracy
loss_v2, acc_v2 = ...
print(f"Accuracy modelo v2: {acc_v2:.4f}")

### <FONT SIZE=3 COLOR="blue"> Ejercicio 3 </FONT>
Construye una red neuronal para un problema de **regresión**. Usaremos el dataset **California Housing** (incluido en `sklearn.datasets`), que contiene información de viviendas en California. La variable objetivo es el **precio mediano** (en cientos de miles de USD) y los predictores son 8 features como ingreso mediano del bloque, edad de la casa, número de habitaciones, latitud, longitud, etc.

**Diferencias clave respecto a clasificación:**
- La capa de salida tiene **1 neurona** y **sin activación** (lineal).
- La función de pérdida es **`mse`** (no `binary_crossentropy`).
- La métrica principal es **`mae`** o **MSE**, no accuracy.

In [ ]:
# Cargamos el dataset California Housing (precios de vivienda en California)
from sklearn.datasets import fetch_california_housing

california = fetch_california_housing()
X_apt = california.data       # 8 features
y_apt = california.target     # precio mediano (en cientos de miles de USD)

print("Forma de X:", X_apt.shape)
print("Features:", california.feature_names)
print("\nPrimeras 5 filas:")
print(pd.DataFrame(X_apt, columns=california.feature_names).head())

# Train / test
X_tr, X_te, y_tr, y_te = train_test_split(X_apt, y_apt, test_size=0.2, random_state=42)

# Escalado de X (¡siempre escalar en redes neuronales!)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)

# TU CÓDIGO AQUÍ:
# 1. Define una red con 2 capas ocultas (32 y 16 neuronas, ReLU) y 1 capa de salida lineal
modelo_reg = Sequential([
    ...
])

# 2. Compila con optimizador 'adam', loss 'mse', y metrics ['mae']
modelo_reg.compile(...)

# 3. Entrena 50 epochs, batch_size=32, validation_split=0.2
historia_reg = ...

# 4. Evalúa y calcula r2_score en test
y_pred_apt = modelo_reg.predict(X_te, verbose=0).flatten()
r2 = r2_score(y_te, y_pred_apt)
print(f"\nR² del modelo de regresión: {r2:.4f}")

### <FONT SIZE=3 COLOR="blue"> Ejercicio 4 — Reto </FONT>
Compara el desempeño de **tres modelos** sobre el dataset `make_moons` con `noise=0.30`:

1. Una **regresión logística** (modelo lineal de `sklearn`).
2. Una **red neuronal con 1 sola capa oculta de 4 neuronas**.
3. Una **red neuronal con 2 capas ocultas de 32 neuronas cada una**.

Para cada uno, reporta la **accuracy en test** y comenta cuál crees que es mejor para este problema y por qué.

In [ ]:
# TU CÓDIGO AQUÍ:

from sklearn.linear_model import LogisticRegression

# Generamos los datos
X, y = make_moons(n_samples=1000, noise=0.30, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr); X_te_s = sc.transform(X_te)

# 1. Regresión logística
log_reg = ...
acc_logreg = ...
print(f"Accuracy Regresión Logística: {acc_logreg:.4f}")

# 2. Red pequeña (1 capa oculta de 4 neuronas)
mod_small = Sequential([
    ...
])
mod_small.compile(...)
mod_small.fit(...)
acc_small = ...
print(f"Accuracy Red pequeña:         {acc_small:.4f}")

# 3. Red grande (2 capas ocultas de 32 neuronas)
mod_big = Sequential([
    ...
])
mod_big.compile(...)
mod_big.fit(...)
acc_big = ...
print(f"Accuracy Red grande:          {acc_big:.4f}")

---
# <FONT SIZE=5 COLOR="purple"> 6. Conclusiones de la sesión </FONT>

En esta sesión sentamos las bases del **Deep Learning**:

<FONT COLOR="green" SIZE=3>1. Neurona y Perceptrón:</FONT> entendimos la unidad mínima de cómputo: una suma ponderada $z = \sum w_i x_i + b$ seguida de una función de activación.

<FONT COLOR="green" SIZE=3>2. Funciones de activación:</FONT> introducen la **no linealidad** indispensable. **ReLU** para capas ocultas, **Sigmoide** para clasificación binaria, **Softmax** para multiclase.

<FONT COLOR="green" SIZE=3>3. Backpropagation:</FONT> el algoritmo que permite a la red **aprender** ajustando los pesos en la dirección opuesta al gradiente de la función de pérdida (descenso del gradiente).

<FONT COLOR="green" SIZE=3>4. Keras + TensorFlow:</FONT> el flujo `Sequential` → `compile` → `fit` → `evaluate` nos permite construir redes potentes con muy pocas líneas de código.

---

<FONT SIZE=3 COLOR="purple">**¿Qué viene después?**</FONT>

- **Redes Convolucionales (CNNs):** especializadas en imágenes.
- **Redes Recurrentes (RNNs / LSTM):** especializadas en secuencias y texto.
- **Transformers:** la arquitectura detrás de ChatGPT, BERT, etc.
- **Regularización avanzada:** Batch Normalization, Early Stopping, Data Augmentation.

¡Nos vemos en la próxima sesión! 🚀